In [0]:
# Load your Delta table
df = spark.read.table("default.amazon_sales_data_2025")

# Show a preview
df.display()
df.show()

Order ID,Date,Product,Category,Price,Quantity,Total Sales,Customer Name,Customer Location,Payment Method,Status
ORD0001,2025-03-14,Running Shoes,Footwear,60,3,180,Emma Clark,New York,Debit Card,Cancelled
ORD0002,2025-03-20,Headphones,Electronics,100,4,400,Emily Johnson,San Francisco,Debit Card,Pending
ORD0003,2025-02-15,Running Shoes,Footwear,60,2,120,John Doe,Denver,Amazon Pay,Cancelled
ORD0004,2025-02-19,Running Shoes,Footwear,60,3,180,Olivia Wilson,Dallas,Credit Card,Pending
ORD0005,2025-03-10,Smartwatch,Electronics,150,3,450,Emma Clark,New York,Debit Card,Pending
ORD0006,2025-03-14,T-Shirt,Clothing,20,1,20,John Doe,Dallas,Credit Card,Pending
ORD0007,2025-03-18,Smartwatch,Electronics,150,4,600,Emma Clark,Houston,PayPal,Completed
ORD0008,2025-03-02,Smartphone,Electronics,500,1,500,Sophia Miller,Miami,PayPal,Completed
ORD0009,2025-03-08,T-Shirt,Clothing,20,3,60,Sophia Miller,Boston,PayPal,Completed
ORD0010,2025-03-12,Smartphone,Electronics,500,1,500,Emily Johnson,San Francisco,Credit Card,Cancelled


+--------+----------+---------------+---------------+-----+--------+-----------+-------------+-----------------+--------------+---------+
|Order ID|      Date|        Product|       Category|Price|Quantity|Total Sales|Customer Name|Customer Location|Payment Method|   Status|
+--------+----------+---------------+---------------+-----+--------+-----------+-------------+-----------------+--------------+---------+
| ORD0001|2025-03-14|  Running Shoes|       Footwear|   60|       3|        180|   Emma Clark|         New York|    Debit Card|Cancelled|
| ORD0002|2025-03-20|     Headphones|    Electronics|  100|       4|        400|Emily Johnson|    San Francisco|    Debit Card|  Pending|
| ORD0003|2025-02-15|  Running Shoes|       Footwear|   60|       2|        120|     John Doe|           Denver|    Amazon Pay|Cancelled|
| ORD0004|2025-02-19|  Running Shoes|       Footwear|   60|       3|        180|Olivia Wilson|           Dallas|   Credit Card|  Pending|
| ORD0005|2025-03-10|     Smartwat

In [0]:
df_clean = df.dropna().dropDuplicates()

In [0]:
from pyspark.sql.functions import col, row_number,  month,dayofweek
from pyspark.sql.window import Window


#now here adding a month and weekday
df_transformed = df_clean.withColumn("Month",month("Date")).withColumn("Weekday",dayofweek("Date"))

# now lets rank top selling products as per category 

windowSpec = Window.partitionBy("Category").orderBy(col("Total Sales").desc())

df_ranked = df_transformed.withColumn("Sales Rank",row_number().over(windowSpec))
df_ranked.display()

Order ID,Date,Product,Category,Price,Quantity,Total Sales,Customer Name,Customer Location,Payment Method,Status,Month,Weekday,Sales Rank
ORD0113,2025-03-19,Book,Books,15,5,75,David Lee,San Francisco,Debit Card,Pending,3,4,1
ORD0193,2025-03-30,Book,Books,15,5,75,David Lee,Chicago,Amazon Pay,Pending,3,1,2
ORD0184,2025-02-17,Book,Books,15,5,75,Daniel Harris,Miami,Debit Card,Pending,2,2,3
ORD0025,2025-03-02,Book,Books,15,5,75,Sophia Miller,Seattle,Amazon Pay,Completed,3,1,4
ORD0060,2025-03-12,Book,Books,15,5,75,Jane Smith,Dallas,Credit Card,Pending,3,4,5
ORD0101,2025-02-20,Book,Books,15,5,75,John Doe,Denver,PayPal,Pending,2,5,6
ORD0177,2025-03-14,Book,Books,15,5,75,David Lee,San Francisco,Credit Card,Pending,3,6,7
ORD0097,2025-03-25,Book,Books,15,5,75,Olivia Wilson,Chicago,Amazon Pay,Pending,3,3,8
ORD0201,2025-02-03,Book,Books,15,4,60,Michael Brown,San Francisco,Credit Card,Completed,2,2,9
ORD0150,2025-02-08,Book,Books,15,4,60,Daniel Harris,Chicago,Gift Card,Cancelled,2,7,10


In [0]:
df_cleaned = df_ranked
for col_name in df_ranked.columns:
    df_cleaned = df_cleaned.withColumnRenamed(col_name, col_name.strip().replace(" ", "_"))

# Now save to Delta
df_cleaned.write.format("delta").mode("overwrite").saveAsTable("default.amazon_sales_ranked_2025")


In [0]:
%sql
SELECT * FROM default.amazon_sales_ranked_2025


Order_ID,Date,Product,Category,Price,Quantity,Total_Sales,Customer_Name,Customer_Location,Payment_Method,Status,Month,Weekday,Sales_Rank
ORD0113,2025-03-19,Book,Books,15,5,75,David Lee,San Francisco,Debit Card,Pending,3,4,1
ORD0193,2025-03-30,Book,Books,15,5,75,David Lee,Chicago,Amazon Pay,Pending,3,1,2
ORD0184,2025-02-17,Book,Books,15,5,75,Daniel Harris,Miami,Debit Card,Pending,2,2,3
ORD0025,2025-03-02,Book,Books,15,5,75,Sophia Miller,Seattle,Amazon Pay,Completed,3,1,4
ORD0060,2025-03-12,Book,Books,15,5,75,Jane Smith,Dallas,Credit Card,Pending,3,4,5
ORD0101,2025-02-20,Book,Books,15,5,75,John Doe,Denver,PayPal,Pending,2,5,6
ORD0177,2025-03-14,Book,Books,15,5,75,David Lee,San Francisco,Credit Card,Pending,3,6,7
ORD0097,2025-03-25,Book,Books,15,5,75,Olivia Wilson,Chicago,Amazon Pay,Pending,3,3,8
ORD0201,2025-02-03,Book,Books,15,4,60,Michael Brown,San Francisco,Credit Card,Completed,2,2,9
ORD0150,2025-02-08,Book,Books,15,4,60,Daniel Harris,Chicago,Gift Card,Cancelled,2,7,10


In [0]:
df_ranked.columns


['Order ID',
 'Date',
 'Product',
 'Category',
 'Price',
 'Quantity',
 'Total Sales',
 'Customer Name',
 'Customer Location',
 'Payment Method',
 'Status',
 'Month',
 'Weekday',
 'Sales Rank']

In [0]:
# Replace spaces with underscores in all column names
for col_name in df_ranked.columns:
    df_ranked = df_ranked.withColumnRenamed(col_name, col_name.strip().replace(" ", "_"))


In [0]:
df_ranked.columns


['Order_ID',
 'Date',
 'Product',
 'Category',
 'Price',
 'Quantity',
 'Total_Sales',
 'Customer_Name',
 'Customer_Location',
 'Payment_Method',
 'Status',
 'Month',
 'Weekday',
 'Sales_Rank']

In [0]:
df_customers = df_ranked.groupBy("Customer_Name").agg(
    {"Total_Sales": "sum", "Order_ID": "count"}
).withColumnRenamed("sum(Total_Sales)", "Total_Spent") \
 .withColumnRenamed("count(Order_ID)", "Total_Orders")

df_customers.display()


Customer_Name,Total_Spent,Total_Orders
John Doe,26870,26
Daniel Harris,18945,23
Chris White,18885,22
Emma Clark,29700,32
David Lee,22665,26
Emily Johnson,23475,22
Michael Brown,22655,24
Sophia Miller,13295,16
Jane Smith,31185,30
Olivia Wilson,36170,29


In [0]:
df_customers.write.format("delta").mode("overwrite").saveAsTable("default.amazon_customer_stats_2025")
